## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
# Time window (wave 1b): AM 06:00-09:00 is the base and the default; PM 16:00-19:00 and midday 10:00-14:00 with the environment variable NOFIT_PERIOD
PERIOD = os.environ.get('NOFIT_PERIOD', 'AM')
HOURS, SUF = {'AM': ([6, 7, 8], ''), 'PM': ([16, 17, 18], '_pm'), 'MD': ([10, 11, 12, 13], '_md')}[PERIOD]
PLABEL = {'AM': 'AM 06:00-09:00', 'PM': 'PM 16:00-19:00', 'MD': 'midday 10:00-14:00'}[PERIOD]
W = {'AM': '06-09', 'PM': '16-19', 'MD': '10-14'}[PERIOD]
PSUF = '' if PERIOD == 'AM' else '_' + PERIOD            # suffix of the validation output files of the other periods
def vfile(name): return f'{OUT}/{name}{PSUF}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'period': PERIOD, 'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name.replace(".png", PSUF + ".png")}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

# Ministry of Transport validation, stage 1: inputs and trip distribution (T4, T5, T7)

**Question.** Does the survey-based matrix pass the guideline's distribution checks against data that are
independent of the matrix: the survey's own reported trip distances (T4), the cellular-phone matrix (T5) and the
RavKav smart-card journeys (T7)?

**Frozen version.** The 23 September 2026 rebuild (step 15 on RavKav's own alightings; steps 16-36 rerun).
Nothing in the chain is changed here: a miss is reported with its reason, not tuned away.

**Criteria (guideline draft 6).** T4: the two trip-length distributions pass the Kolmogorov-Smirnov test. T5:
coincidence ratio above 0.6 against the cellular matrix. T7: R2 above 0.85 and a slope of 0.9-1.1 (regression
through the origin) at super-zone level against the RavKav / on-board matrix.

## 1. Zones, keys and the survey trips

In [2]:
TRIPS = 'Input/THS_2017-2018/trips_ths_2017.xlsx'
CELL = 'Input/Matrices/AvgDayHourlyTrips201819_1270_weekday_v1.csv'
KEYS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
need(TRIPS, CELL, KEYS, 'Input/TAZ_2636_Keys.xlsx', 'Input/Submatrix_tazs.xlsx', 'Input/TAZ_GSnew.csv')

zon = pd.read_csv('Input/Zonal_2020.csv', encoding='cp1255').set_index('TAZ_ID')
TAZ = [int(c) for c in pd.read_csv('Output/ths2017/two_mode/car_taz.csv', index_col=0).columns]     # the 778 study TAZs
_k = pd.read_csv('Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv', encoding='windows-1255')[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW']
taz_sz = _k.reindex(TAZ).astype(int)                                                               # TAZ -> super-zone (36), the key the chain uses
print('TAZs whose super-zone differs between the keys table and Zonal_2020:', int((taz_sz.values != zon.loc[TAZ, 'SZ_NEW'].astype(int).values).sum()))
gs_key = pd.read_csv('Input/TAZ_GSnew.csv').set_index('TAZ')['GS']; taz_gs = gs_key.reindex(TAZ)
sub = pd.read_excel('Input/Submatrix_tazs.xlsx'); taz_area = sub.set_index('TAZ')['AggAreaCode']       # 28 sub-areas (205 TAZs)

keys_raw = pd.read_csv(KEYS, encoding='windows-1255')
kd = keys_raw[['TAZ_1270', 'TAZ_NUMBER']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int}).drop_duplicates('TAZ_NUMBER')
taz_1250 = kd.set_index('TAZ_NUMBER')['TAZ_1270'].reindex(TAZ)
assert taz_1250.notna().all(), 'every study TAZ must map to a 1250 zone'
Z1250 = sorted(taz_1250.unique())
print(f'{len(TAZ)} study TAZs, {len(taz_sz.unique())} super-zones, {len(Z1250)} 1250-zones')

def dominant(group_of_taz):
    """Group (super-zone, GS) of each 1250 zone = the group holding most of its 2020 population (employment if none)."""
    d = pd.DataFrame({'z': taz_1250.values, 'g': group_of_taz.values, 'pop': zon.loc[TAZ, 'POPULATION'].values,
                      'emp': zon.loc[TAZ, 'EMPL_TOT'].values}, index=TAZ).dropna(subset=['g'])
    out = {}
    for z, grp in d.groupby('z'):
        w = grp.groupby('g')['pop'].sum()
        if w.sum() <= 0: w = grp.groupby('g')['emp'].sum()
        out[z] = int(w.idxmax()) if w.sum() > 0 else int(grp['g'].iloc[0])
    return pd.Series(out)
sz_of_1250 = dominant(taz_sz); gs_of_1250 = dominant(taz_gs)

# --- survey trips: exactly the extraction of THS_2017_trips_matrices (origin = previous stop, departure hour = its Dep_h)
df = pd.read_excel(TRIPS)
assert set(df['SurveyDay'].unique()) == {1, 2}
d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy()
g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1); d['trip_dep_h'] = g['Dep_h'].shift(1); d['o_act'] = g['mainActivity'].shift(1)
trips = d.dropna(subset=['origin', 'actTaz', 'trip_dep_h']).copy()
k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636').set_index('TAZ_2636')['TAZ_1250']
trips['o1250'] = trips['origin'].map(k26); trips['d1250'] = trips['actTaz'].map(k26)
trips['in_study'] = trips['o1250'].isin(Z1250) & trips['d1250'].isin(Z1250)
trips['sz_o'] = trips['o1250'].map(sz_of_1250); trips['sz_d'] = trips['d1250'].map(sz_of_1250)
trips['gs_o'] = trips['o1250'].map(gs_of_1250); trips['gs_d'] = trips['d1250'].map(gs_of_1250)
CAR, BUS, TAXI, RAIL = [10, 11], [3, 5], [4, 8], [7]        # trips-file codes confirmed on 23 Sep 2026 (METHODOLOGY 6ae)
def mode_group(m):
    return 'car' if m in CAR else 'bus' if m in BUS else 'taxi' if m in TAXI else 'rail' if m in RAIL else 'other'
trips['grp'] = trips['mainmode'].map(mode_group)
trips['am'] = trips['trip_dep_h'].isin(HOURS)                  # the window of the run (AM by default)
S = trips[trips['in_study']]
print(f'trips with both ends inside the study area: {len(S):,} sampled ({S.new_wf.sum()/2:,.0f} expanded per average day); {PLABEL}: {S[S.am].new_wf.sum()/2:,.0f}')
chk = pd.read_csv('Output/ths2017/matrix_avg_ALL.csv', index_col=0)
print(f'check against the saved 2636-zone matrix_avg_ALL.csv (both ends in the 2636 system, all trips): {chk.values.sum():,.0f} vs rebuilt window, both ends in study area: {S[S.am].new_wf.sum()/2:,.0f}')

TAZs whose super-zone differs between the keys table and Zonal_2020: 142
778 study TAZs, 36 super-zones, 396 1250-zones


trips with both ends inside the study area: 106,770 sampled (8,513,341 expanded per average day); PM 16:00-19:00: 1,684,480
check against the saved 2636-zone matrix_avg_ALL.csv (both ends in the 2636 system, all trips): 2,184,632 vs rebuilt window, both ends in study area: 1,684,480


In [3]:
n_diff = int((taz_sz.values != zon.loc[TAZ, 'SZ_NEW'].astype(int).values).sum())
add_summary('T1', '3.1', 'super-zone key consistency', 'TAZ', 'one agreed zone system',
            f'{n_diff} of {len(TAZ)} TAZs carry a different super-zone in Zonal_2020.csv (SZ_NEW) than in the 1270 keys table', 'finding',
            'The chain uses the keys table. The zone system must be fixed with the steering group before any comparison with CBS or the national model (guideline 3.1).', 'Input/Zonal_2020.csv vs Input/Matrices/1270_...keys.csv')

## 2. T5 - the survey matrix against the cellular matrix (guideline 3.5)

**Test.** Coincidence ratio (CR = sum of the cell-wise minima over the sum of the maxima of the two matrices after
each is turned into proportions) between the survey and the cellular-phone matrix at super-zone level, for the
AM period (06:00-09:00) and for the whole day. **Criterion:** CR above 0.6. The guideline asks for the comparison
with the survey and with the cellular survey; the survey is the matrix itself here, so the cellular one is the
independent reference. Both matrices are aggregated to super-zones through the same 1250-zone key (dominant
super-zone by 2020 population), so the aggregation cannot favour either. Survey day 1 against day 2 is the
repeatability reference. The off-diagonal variant removes intra-zonal cells, where most of the known divergence sits.

In [4]:
cell = pd.read_csv(CELL)
hc = [f'h{i}' for i in range(24)]
cell = cell[cell['fromZone'].isin(Z1250) & cell['ToZone'].isin(Z1250)].copy()
cell['am'] = cell[[f'h{h}' for h in HOURS]].sum(axis=1); cell['day'] = cell[hc].sum(axis=1)
print(f'cellular, both ends inside the study area: {PERIOD} {cell.am.sum():,.0f}, day {cell.day.sum():,.0f} trips')

def to_mat(df, o, d, w, idx):
    return df.groupby([o, d])[w].sum().unstack().reindex(index=idx, columns=idx, fill_value=0).fillna(0)

def level_maps(level):
    return {'SZ': ('sz_o', 'sz_d', sz_of_1250, sorted(sz_of_1250.unique())),
            'GS': ('gs_o', 'gs_d', gs_of_1250, sorted(gs_of_1250.unique()))}[level]

rows = []; MATS = {}
for level in ['SZ', 'GS']:
    o, dd, zmap, idx = level_maps(level)
    c = cell.assign(zo=cell['fromZone'].map(zmap), zd=cell['ToZone'].map(zmap))
    for period, hours, cw in [(PLABEL, HOURS, 'am'), ('whole day', None, 'day')]:
        sel = S if hours is None else S[S['trip_dep_h'].isin(hours)]
        cellular = to_mat(c, 'zo', 'zd', cw, idx).values
        surv = {}
        for nm, sub_ in [('survey', sel), ('survey day 1', sel[sel.SurveyDay == 1]), ('survey day 2', sel[sel.SurveyDay == 2])]:
            m = to_mat(sub_, o, dd, 'new_wf', idx).values
            surv[nm] = m / 2 if nm == 'survey' else m
        MATS[(level, period)] = {'cellular': cellular, **surv}
        off = ~np.eye(len(idx), dtype=bool)
        for nm, A, B in [('survey vs cellular', surv['survey'], cellular), ('survey day 1 vs day 2', surv['survey day 1'], surv['survey day 2'])]:
            rows.append({'level': level, 'period': period, 'pair': nm, 'cells': len(idx) ** 2,
                         'CR (all cells)': vm.coincidence_ratio(A, B), 'CR (off-diagonal)': vm.coincidence_ratio(A[off], B[off]),
                         'volume ratio B / A': B.sum() / A.sum(), 'R2': vm.r2_correl(A, B)})
t5 = pd.DataFrame(rows); t5.to_csv(vfile('T5_coincidence_ratio'), index=False, float_format='%.4f')
t5.round(3)

cellular, both ends inside the study area: PM 670,115, day 3,384,827 trips


,level,period,pair,cells,CR (all cells),CR (off-diagonal),volume ratio B / A,R2
0,SZ,PM 16:00-19:00,survey vs cellular,1296,0.504,0.543,0.398,0.904
1,SZ,PM 16:00-19:00,survey day 1 vs day 2,1296,0.788,0.595,0.936,0.980
2,SZ,whole day,survey vs cellular,1296,0.471,0.625,0.398,0.894
3,SZ,whole day,survey day 1 vs day 2,1296,0.893,0.767,0.962,0.995
4,GS,PM 16:00-19:00,survey vs cellular,625,0.605,0.624,0.398,0.951
5,GS,PM 16:00-19:00,survey day 1 vs day 2,625,0.849,0.628,0.936,0.994
6,GS,whole day,survey vs cellular,625,0.548,0.715,0.398,0.941
7,GS,whole day,survey day 1 vs day 2,625,0.926,0.791,0.962,0.999


In [5]:
for _, r in t5[(t5.pair == 'survey vs cellular')].iterrows():
    ok = r['CR (all cells)'] > vm.CRIT['CR_min']
    add_summary('T5', '3.5', f'matrix vs cellular, {r["period"]}', r['level'], 'CR > 0.6',
                f'CR {r["CR (all cells)"]:.2f} (off-diagonal {r["CR (off-diagonal)"]:.2f}); survey day 1 vs day 2: '
                f'{t5[(t5.level == r["level"]) & (t5.period == r["period"]) & (t5.pair == "survey day 1 vs day 2")]["CR (all cells)"].iloc[0]:.2f}',
                vm.verdict(ok, explained=True),
                'Different trip definitions: the cellular matrix has far fewer short trips (survey 3.6 x the volume; 47 % of survey trips stay '
                'inside one cellular zone against 14 %), so the diagonal dominates the gap; see METHODOLOGY 6j-6l.', 'survey 2017/18 vs cellular 2018/19')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), facecolor='white')
for ax, per in zip(axes, [PLABEL, 'whole day']):
    M = MATS[('SZ', per)]
    scatter_origin(ax, M['cellular'].ravel(), M['survey'].ravel(), f'Super-zone OD cells, {per}', 'cellular matrix (trips)', 'survey matrix (trips)', logscale=True)
save_fig(fig, 'mot_T5_survey_vs_cellular.png')

**Reading.** The coincidence ratio is stricter than the cosine reported earlier (METHODOLOGY 6j): it needs the *shape* of every cell
to agree, not the largest cells only. The table gives the result for all cells and without the diagonal, next to the survey's own
repeatability.

## 3. T4 - trip-length distribution (guideline 3.5)

**Test.** Kolmogorov-Smirnov distance between the distribution of trip lengths in the matrix and the survey's own *reported*
trip distance (`TrvlDist`, kilometres, measured by the survey app), for the car and the transit layers. **Criterion:** the two
distributions pass the KS test (D below the critical value at the 5 % level).

The matrix has no trip distance, so a proxy is used: the straight-line distance between zone centroids, with the
intra-zonal distance set to 0.52 x sqrt(zone area) (the mean distance between two random points of a square).
It is evaluated at the TAZ level of the matrix and at the 1250-zone level, where the TAZ allocation (a population /
employment split, not an observation) does not enter. Because a straight line is shorter than the path the survey
measured, a third variant multiplies the proxy by the survey's own median detour factor (reported distance over
straight-line distance on inter-zonal trips). The critical value uses the Kish effective sample size of the survey weights
for both samples (expansion weights would otherwise make every difference significant).
Purpose and sector cannot be tested: the matrices carry neither, so those splits are given for the survey only.

In [6]:
import geopandas as gpd
shp = gpd.read_file('Input/TAZ_North/TAZ_North.shp').drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER').loc[TAZ]
cx = shp.geometry.centroid; taz_xy = pd.DataFrame({'x': cx.x, 'y': cx.y, 'area': shp.geometry.area}, index=TAZ)
pop = zon.loc[TAZ, 'POPULATION'].clip(lower=0) + 1e-9
z1250 = pd.DataFrame({'z': taz_1250.values, 'x': taz_xy.x.values * pop.values, 'y': taz_xy.y.values * pop.values, 'w': pop.values, 'area': taz_xy.area.values}, index=TAZ)
zc = z1250.groupby('z').agg(x=('x', 'sum'), y=('y', 'sum'), w=('w', 'sum'), area=('area', 'sum'))
zc['x'] /= zc['w']; zc['y'] /= zc['w']; zc = zc.loc[Z1250]

def dist_matrix(xy, area):
    X = xy[['x', 'y']].values; D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)) / 1000.0
    np.fill_diagonal(D, 0.52 * np.sqrt(area.values) / 1000.0)
    return D
D_taz = dist_matrix(taz_xy, taz_xy['area']); D_1250 = dist_matrix(zc, zc['area'])
d1250 = pd.DataFrame(D_1250, index=Z1250, columns=Z1250)

car_taz = pd.read_csv(f'Output/ths2017/two_mode{SUF}/car_taz.csv', index_col=0); car_taz.index = car_taz.index.astype(int); car_taz.columns = car_taz.columns.astype(int)
tr_taz = pd.read_csv(f'Output/ths2017/two_mode{SUF}/transit_survey_taz.csv', index_col=0); tr_taz.index = tr_taz.index.astype(int); tr_taz.columns = tr_taz.columns.astype(int)
car_taz = car_taz.reindex(index=TAZ, columns=TAZ, fill_value=0).values; tr_taz = tr_taz.reindex(index=TAZ, columns=TAZ, fill_value=0).values
print(f'matrices (2018, survey only, {PERIOD}): car {car_taz.sum():,.0f}, transit {tr_taz.sum():,.0f}')

_all = trips[trips.in_study & trips.am & trips.grp.isin(['car', 'bus', 'taxi', 'rail'])]
print(f'{PERIOD} study-area trips by car / bus / taxi / rail: {len(_all):,} sampled; with a reported distance: {_all.TrvlDist.notna().mean():.1%} (weighted {np.average(_all.TrvlDist.notna(), weights=_all.new_wf):.1%})')
S1 = _all[_all.TrvlDist.notna()].copy()
S1['layer'] = np.where(S1.grp == 'car', 'car', 'transit'); S1['w'] = S1.new_wf / 2
S1['straight'] = [d1250.at[a, b] for a, b in zip(S1.o1250, S1.d1250)]
inter = S1[S1.o1250 != S1.d1250]
detour = float((inter.TrvlDist / inter.straight).median())
print(f'survey trips with a reported distance: {len(S1):,}; median detour factor (reported / straight-line, inter-zonal) {detour:.2f}')

def agg_1250(M):
    ix = pd.Series(taz_1250.values, index=TAZ)
    return pd.DataFrame(M, index=TAZ, columns=TAZ).groupby(ix.values).sum().T.groupby(ix.values).sum().T.reindex(index=Z1250, columns=Z1250, fill_value=0).values

rows = []; CDF = {}
for layer, M in [('car', car_taz), ('transit', tr_taz)]:
    sv = S1[S1.layer == layer]; n_eff = vm.kish_n(sv.w)
    M1250 = agg_1250(M)
    variants = [('TAZ proxy', D_taz, M, None), ('1250-zone proxy', D_1250, M1250, None),
                ('1250-zone proxy x detour', D_1250 * detour, M1250, None),
                ('1250-zone proxy, off-diagonal only', D_1250, M1250, 'off')]
    for name, D, MM, kind in variants:
        w = MM.copy(); s = sv
        if kind == 'off':
            np.fill_diagonal(w, 0); s = sv[sv.o1250 != sv.d1250]
        sel = w.ravel() > 0
        r = vm.ks_weighted(s.TrvlDist.values, s.w.values, D.ravel()[sel], w.ravel()[sel], n1=n_eff, n2=n_eff)
        wmed = lambda x, ww: float(x[np.argsort(x)][np.searchsorted(np.cumsum(ww[np.argsort(x)]), 0.5 * ww.sum())])
        rows.append({'layer': layer, 'model distance proxy': name, 'survey trips (sampled)': len(s), 'n effective': n_eff, 'KS D': r['D'], 'critical (5 %)': r['critical'],
                     'pass': r['pass'], 'median km, survey reported': wmed(s.TrvlDist.values, s.w.values), 'median km, matrix': wmed(D.ravel()[sel], w.ravel()[sel]),
                     'mean km, survey reported': float(np.average(s.TrvlDist, weights=s.w)), 'mean km, matrix': float(np.average(D.ravel()[sel], weights=w.ravel()[sel]))})
        CDF[(layer, name)] = (s.TrvlDist.values, s.w.values, D.ravel()[sel], w.ravel()[sel])
t4 = pd.DataFrame(rows); t4.to_csv(vfile('T4_trip_length_KS'), index=False, float_format='%.4f'); t4.round(3)

matrices (2018, survey only, PM): car 1,198,961, transit 91,449
PM study-area trips by car / bus / taxi / rail: 15,831 sampled; with a reported distance: 100.0% (weighted 100.0%)
survey trips with a reported distance: 15,831; median detour factor (reported / straight-line, inter-zonal) 0.97


,layer,model distance proxy,survey trips (sampled),n effective,KS D,critical (5 %),pass,"median km, survey reported","median km, matrix","mean km, survey reported","mean km, matrix"
0,car,TAZ proxy,14507,9733.684,0.169,0.019,False,2.096,2.915,5.812,6.408
1,car,1250-zone proxy,14507,9733.684,0.200,0.019,False,2.096,2.811,5.812,6.403
2,car,1250-zone proxy x detour,14507,9733.684,0.193,0.019,False,2.096,2.731,5.812,6.221
3,car,"1250-zone proxy, off-diagonal only",10343,9733.684,0.090,0.019,False,3.964,4.546,7.946,8.394
4,transit,TAZ proxy,1324,829.911,0.070,0.067,False,3.122,3.372,7.121,7.326
5,transit,1250-zone proxy,1324,829.911,0.067,0.067,False,3.122,3.340,7.121,7.316
6,transit,1250-zone proxy x detour,1324,829.911,0.060,0.067,True,3.122,3.245,7.121,7.108
7,transit,"1250-zone proxy, off-diagonal only",1195,829.911,0.070,0.067,False,3.507,3.782,7.774,7.968


In [7]:
# survey-only splits (the matrices carry no purpose or sector)
def purpose(r):
    o, dd = r['o_act'], r['mainActivity']
    if o == 1 and dd == 1: return 'HBO'
    if o != 1 and dd != 1: return 'NHB'
    a = dd if o == 1 else o
    return 'HBW' if a in (2, 3) else 'HBE' if a == 4 else 'HBO'
S1['purpose'] = S1.apply(purpose, axis=1)
per = pd.read_csv('Input/THS_2017-2018/PersonsFin2.csv', encoding='utf-8-sig'); hh = pd.read_csv('Input/THS_2017-2018/HHfinal.csv', encoding='utf-8-sig')
SECTOR = {'arab': 'Arab', 'orto': 'Haredi'}                       # decision D2: Jewish non-Haredi = secular + religious + other
hh['sector'] = hh['HHTYPE'].map(SECTOR).fillna('Jewish non-Haredi')
per = per.merge(hh[['HHID', 'sector', 'migzar']], on='HHID', how='left')
S1['INDIVID'] = S1['HHID3'] * 100 + S1['IndID3']
S1 = S1.merge(per[['INDIVID', 'sector']], on='INDIVID', how='left')
def wq(x, w, q):
    o = np.argsort(x.values); return float(x.values[o][np.searchsorted(np.cumsum(w.values[o]), q * w.sum())])
sp = (S1.groupby(['layer', 'purpose']).apply(lambda g: pd.Series({'sampled trips': len(g), 'median km': wq(g.TrvlDist, g.w, .5), 'mean km': np.average(g.TrvlDist, weights=g.w)}), include_groups=False).reset_index())
ss = (S1.groupby(['layer', 'sector']).apply(lambda g: pd.Series({'sampled trips': len(g), 'median km': wq(g.TrvlDist, g.w, .5), 'mean km': np.average(g.TrvlDist, weights=g.w)}), include_groups=False).reset_index())
sp.to_csv(vfile('T4_survey_distance_by_purpose'), index=False, float_format='%.3f'); ss.to_csv(vfile('T4_survey_distance_by_sector'), index=False, float_format='%.3f')
print(sp.round(2).to_string(index=False)); print(ss.round(2).to_string(index=False))

  layer purpose  sampled trips  median km  mean km
    car     HBE          689.0       1.02     4.17
    car     HBO         7173.0       1.68     4.54
    car     HBW         2963.0       6.38    10.54
    car     NHB         3682.0       1.79     4.75
transit     HBE          167.0       6.35    10.97
transit     HBO          542.0       2.18     4.67
transit     HBW          285.0       4.81     9.95
transit     NHB          330.0       3.61     6.78
  layer            sector  sampled trips  median km  mean km
    car              Arab         4673.0       3.02     7.56
    car            Haredi          491.0       1.26     3.42
    car Jewish non-Haredi         9341.0       1.87     4.89
transit              Arab          144.0      11.55    13.24
transit            Haredi          136.0       1.56     5.55
transit Jewish non-Haredi         1043.0       2.86     6.09


In [8]:
for _, r in t4.iterrows():
    if r['model distance proxy'] in ('1250-zone proxy', '1250-zone proxy x detour', 'TAZ proxy'):
        add_summary('T4', '3.5', f'trip-length KS, {r["layer"]} layer', r['model distance proxy'], 'pass KS (D < critical)',
                    f'D {r["KS D"]:.3f} vs critical {r["critical (5 %)"]:.3f}; median {r["median km, matrix"]:.1f} km (matrix) vs {r["median km, survey reported"]:.1f} km (survey reported)',
                    vm.verdict(bool(r['pass']), explained=True),
                    'The matrix distance is a straight line between centroids with a nominal intra-zonal distance; the survey measured path distance. '
                    'Below the 1250-zone level the TAZ allocation is a population / employment split, not an observation.', f'survey 2017/18; matrix 2018 survey-only {PERIOD}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), facecolor='white')
for ax, layer in zip(axes, ['car', 'transit']):
    for name, col, ls in [('1250-zone proxy', BLUE, '-'), ('1250-zone proxy x detour', AQUA, '--'), ('TAZ proxy', PURPLE, ':')]:
        xs, ws, xm, wm = CDF[(layer, name)]
        for x, w, c, lab, l in [(xm, wm, col, f'matrix, {name}', ls)]:
            o = np.argsort(x); ax.plot(x[o], np.cumsum(w[o]) / w.sum(), color=c, ls=l, lw=1.6, label=lab)
    xs, ws, _, _ = CDF[(layer, '1250-zone proxy')]; o = np.argsort(xs)
    ax.plot(xs[o], np.cumsum(ws[o]) / ws.sum(), color=ORANGE, lw=2.2, label='survey, reported distance')
    ax.set_xlim(0, 40); ax.legend(frameon=False, fontsize=8, loc='lower right'); style_ax(ax, f'{layer.capitalize()} trips, {PERIOD}', 'trip length (km)', 'cumulative share of trips')
save_fig(fig, 'mot_T4_trip_length_ks.png')

## 4. T7 - the survey-based transit matrix against the RavKav matrices (guideline 3.5 and 5.2)

**Test.** R2 (square of the correlation), slope of the regression through the origin and the coincidence ratio between the bus
matrix and a smart-card (clearing-house) matrix, cell by cell at super-zone level (35 x 35 cells) and at the 28 sub-areas.
**Criterion:** R2 above 0.85 and slope 0.9-1.1 (CR above 0.6 is reported beside them). The reference is RavKav's May 2022 journeys on
RavKav's own inferred alightings (step 8), and the 2025 layer for the 2022 product. Four survey products are compared: the raw
survey bus matrix (independent of the calibration), the calibrated 2018 matrix and the 2022 layer (which take RavKav volumes where
coverage is credible, so these are checks of direction, not independent validation), and, as a negative control, the RavKav
volumes spread over the on-board survey's destination pattern (step 9). Survey day 1 against day 2 is the repeatability reference.

In [9]:
def load_taz_mat(path, rows=TAZ, cols=TAZ):
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int)
    return m.reindex(index=rows, columns=cols, fill_value=0).fillna(0)
def load_sz_mat(path):
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int)
    SZL = sorted(taz_sz.unique()); return m.reindex(index=SZL, columns=SZL, fill_value=0).fillna(0)
def taz_to_group(M, grp):
    g = grp.reindex(M.index); h = grp.reindex(M.columns)
    out = M.groupby(g.values).sum().T.groupby(h.values).sum().T
    return out
SZL = sorted(taz_sz.unique())
rk22 = load_taz_mat('Output/bus/bus_od_taz_avg.csv' if PERIOD == 'AM' else f'Output/bus/periods/bus_od_taz_avg_{PERIOD}.csv')
REF = {'RavKav 2022 (own alightings)': rk22}
if PERIOD == 'AM':          # the on-board pattern and the 2025 layer exist for the AM window only
    rk22_ob = load_taz_mat('Output/bus/bus_od_taz_new.csv'); rk25 = load_taz_mat('Output/ravkav_2025/bus_od_taz_2025.csv')
    REF.update({'RavKav 2022 (on-board pattern)': rk22_ob, 'RavKav 2025 (own-alighting pattern)': rk25})
print('RavKav inside the 778 TAZs: ' + '; '.join(f'{k} {v.values.sum():,.0f}' for k, v in REF.items()))
REF_SZ = {k: taz_to_group(v, taz_sz).reindex(index=SZL, columns=SZL, fill_value=0).values for k, v in REF.items()}
sub_t = [t for t in TAZ if t in taz_area.index]
REF_AR = {k: taz_to_group(v.loc[sub_t, sub_t], taz_area).values for k, v in REF.items()}
AREAL = sorted(taz_area.loc[sub_t].unique())

bus_raw = load_sz_mat(f'Output/ths2017/two_mode{SUF}/bus_survey_sz.csv').values
bus_cal = load_sz_mat(f'Output/ths2017/two_mode{SUF}/bus_calibrated_sz.csv').values
bus_22 = load_sz_mat(f'Output/ths2017/three_mode_2022{SUF}/bus_2022_sz.csv').values
def tazmat_area(path): return taz_to_group(load_taz_mat(path).loc[sub_t, sub_t], taz_area).values
AR = {'survey bus 2018 (raw)': tazmat_area(f'Output/ths2017/two_mode{SUF}/bus_survey_taz.csv'),
      'bus 2018 calibrated': tazmat_area(f'Output/ths2017/two_mode{SUF}/bus_calibrated_taz.csv'),
      'bus 2022 layer': tazmat_area(f'Output/ths2017/three_mode_2022{SUF}/bus_2022_taz.csv')}
# repeatability reference: survey bus trips (modes 3 and 5), the window of the run, day 1 vs day 2, super-zone level
bsel = S[S.am & (S.grp == 'bus')]
b1 = to_mat(bsel[bsel.SurveyDay == 1], 'sz_o', 'sz_d', 'new_wf', SZL).values; b2 = to_mat(bsel[bsel.SurveyDay == 2], 'sz_o', 'sz_d', 'new_wf', SZL).values
SURV = {'survey bus 2018 (raw)': bus_raw, 'bus 2018 calibrated': bus_cal, 'bus 2022 layer': bus_22}
PAIRS = [p_ for p_ in [('survey bus 2018 (raw)', 'RavKav 2022 (own alightings)'), ('bus 2018 calibrated', 'RavKav 2022 (own alightings)'),
         ('bus 2022 layer', 'RavKav 2022 (own alightings)'), ('bus 2022 layer', 'RavKav 2025 (own-alighting pattern)'),
         ('survey bus 2018 (raw)', 'RavKav 2022 (on-board pattern)')] if p_[1] in REF]
rows = []; CELLS = {}
for level, survd, refd, n in [(f'super-zone ({len(SZL)})', SURV, REF_SZ, len(SZL)), ('sub-area (28)', AR, REF_AR, len(AREAL))]:
    off = ~np.eye(n, dtype=bool)
    for a, b in PAIRS:
        A, R = survd[a], refd[b]
        for scope, msk in [('all cells', np.ones((n, n), bool)), ('off-diagonal', off)]:
            fs = vm.fit_stats(R[msk], A[msk])               # obs = RavKav (control), mod = survey-based matrix
            rows.append({'level': level, 'model matrix': a, 'reference': b, 'cells': scope, **fs, 'CR': vm.coincidence_ratio(A[msk], R[msk])})
        if level.startswith('super'): CELLS[(a, b)] = (R.ravel(), A.ravel())
    if level.startswith('super'):
        off_m = ~np.eye(n, dtype=bool)
        for scope, msk in [('all cells', np.ones((n, n), bool)), ('off-diagonal', off_m)]:
            fs = vm.fit_stats(b1[msk], b2[msk]); rows.append({'level': level, 'model matrix': 'survey bus day 2', 'reference': 'survey bus day 1 (repeatability)', 'cells': scope, **fs, 'CR': vm.coincidence_ratio(b2[msk], b1[msk])})
t7 = pd.DataFrame(rows); t7.to_csv(vfile('T7_transit_OD_fit'), index=False, float_format='%.4f')
t7[t7.cells == 'all cells'][['level', 'model matrix', 'reference', 'n', 'obs_total', 'mod_total', 'R2', 'slope_origin', 'RMSE_pct', 'CR']].round(3)

RavKav inside the 778 TAZs: RavKav 2022 (own alightings) 78,490


,level,model matrix,reference,n,obs_total,mod_total,R2,slope_origin,RMSE_pct,CR
0,super-zone (36),survey bus 2018 (raw),RavKav 2022 (own alightings),1296,78490.250,81397.925,0.678,1.127,297.283,0.450
2,super-zone (36),bus 2018 calibrated,RavKav 2022 (own alightings),1296,78490.250,87353.801,0.805,1.236,248.112,0.640
4,super-zone (36),bus 2022 layer,RavKav 2022 (own alightings),1296,78490.250,88817.358,0.772,1.283,286.415,0.628
6,super-zone (36),survey bus day 2,survey bus day 1 (repeatability),1296,84559.476,78236.353,0.732,0.690,291.102,0.489
8,sub-area (28),survey bus 2018 (raw),RavKav 2022 (own alightings),784,22775.750,22389.183,0.735,1.086,257.337,0.363
10,sub-area (28),bus 2018 calibrated,RavKav 2022 (own alightings),784,22775.750,22331.484,0.934,0.942,99.425,0.648
12,sub-area (28),bus 2022 layer,RavKav 2022 (own alightings),784,22775.750,22377.693,0.934,0.943,99.540,0.647


In [10]:
pk = pd.DataFrame({'RavKav 2022 (own alightings), super-zone cells': REF_SZ['RavKav 2022 (own alightings)'].ravel(), 'survey bus 2018 (raw)': bus_raw.ravel(),
                   'bus 2018 calibrated': bus_cal.ravel(), 'bus 2022 layer': bus_22.ravel()})
for k_ in ['RavKav 2025 (own-alighting pattern)', 'RavKav 2022 (on-board pattern)']:
    if k_ in REF_SZ: pk[{'RavKav 2025 (own-alighting pattern)': 'RavKav 2025'}.get(k_, k_)] = REF_SZ[k_].ravel()
pk.insert(0, 'dest_sz', np.tile(SZL, len(SZL))); pk.insert(0, 'orig_sz', np.repeat(SZL, len(SZL)))
pk.to_csv(vfile('T7_matched_cells_superzone'), index=False, float_format='%.3f')
rep_r2 = float(t7[(t7['reference'].str.contains('repeatability')) & (t7.cells == 'all cells')]['R2'].iloc[0])
for _, r in t7[(t7.cells == 'all cells') & (t7.reference.str.startswith('RavKav'))].iterrows():
    ok = (r['R2'] > vm.CRIT['R2_min']) and (vm.CRIT['slope_lo'] <= r['slope_origin'] <= vm.CRIT['slope_hi'])
    control = 'on-board' in r['reference']; is_sz = r['level'].startswith('super')
    why = ('Negative control: the on-board destination pattern is not the pattern the survey shows.' if control else
           ('' if ok else
            f'RavKav records {rk22.values.sum() / bus_raw.sum():.2f} of the survey study-wide in this window (coverage, with the operators, cash fares and un-geocoded stops left to the data provider), so a slope above 1 is expected; '
            + (f'at super-zone level the survey does not repeat itself above R2 {rep_r2:.2f} (day 1 vs day 2), so R2 0.85 is out of reach for a two-day sample; ' if is_sz else '')
            + ('the calibrated layers take RavKav volumes where its coverage is credible, so they are a check of direction.' if 'raw' not in r['model matrix'] else 'the raw survey is independent of the calibration.')))
    add_summary('T7', '3.5 / 5.2', f'{r["model matrix"]} vs {r["reference"]}', r['level'], 'R2 > 0.85 and slope 0.9-1.1',
                f'R2 {r["R2"]:.2f}, slope {r["slope_origin"]:.2f}, RMSE% {r["RMSE_pct"]:.0f}, CR {r["CR"]:.2f}',
                'not applicable (negative control)' if control else vm.verdict(ok, explained=True), why, 'survey 2017/18 and 2022 layer vs RavKav May 2022 / 2025')

fig, axes = plt.subplots(2, 2, figsize=(11, 9.2), facecolor='white')
for ax in axes.ravel()[len([p_ for p_ in PAIRS if 'on-board' not in p_[1]][:4]):]: ax.set_visible(False)
for ax, (a, b) in zip(axes.ravel(), [p_ for p_ in PAIRS if 'on-board' not in p_[1]][:4]):
    R, A = CELLS[(a, b)]
    scatter_origin(ax, R, A, f'{a}\nvs {b}', 'RavKav journeys (observed)', 'survey-based matrix (model)', logscale=True, color=BLUE if 'raw' in a else AQUA)
fig.tight_layout(); save_fig(fig, 'mot_T7_transit_OD_scatter.png')

## Summary of this notebook against the guideline

In [11]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_stage1{PSUF}.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T1,3.1,super-zone key consistency,TAZ,one agreed zone system,142 of 778 TAZs carry a different super-zone i...,finding
1,T5,3.5,"matrix vs cellular, PM 16:00-19:00",SZ,CR > 0.6,CR 0.50 (off-diagonal 0.54); survey day 1 vs d...,miss (explained)
2,T5,3.5,"matrix vs cellular, whole day",SZ,CR > 0.6,CR 0.47 (off-diagonal 0.63); survey day 1 vs d...,miss (explained)
3,T5,3.5,"matrix vs cellular, PM 16:00-19:00",GS,CR > 0.6,CR 0.60 (off-diagonal 0.62); survey day 1 vs d...,pass
4,T5,3.5,"matrix vs cellular, whole day",GS,CR > 0.6,CR 0.55 (off-diagonal 0.71); survey day 1 vs d...,miss (explained)
5,T4,3.5,"trip-length KS, car layer",TAZ proxy,pass KS (D < critical),D 0.169 vs critical 0.019; median 2.9 km (matr...,miss (explained)
6,T4,3.5,"trip-length KS, car layer",1250-zone proxy,pass KS (D < critical),D 0.200 vs critical 0.019; median 2.8 km (matr...,miss (explained)
7,T4,3.5,"trip-length KS, car layer",1250-zone proxy x detour,pass KS (D < critical),D 0.193 vs critical 0.019; median 2.7 km (matr...,miss (explained)
8,T4,3.5,"trip-length KS, transit layer",TAZ proxy,pass KS (D < critical),D 0.070 vs critical 0.067; median 3.4 km (matr...,miss (explained)
9,T4,3.5,"trip-length KS, transit layer",1250-zone proxy,pass KS (D < critical),D 0.067 vs critical 0.067; median 3.3 km (matr...,miss (explained)
